# 01 — Edge-IIoTset Dataset Analysis

**Goal:** Review every row, the Normal/Attack labels and candidate packet/protocol fields.

**Input:** Original `ML-EdgeIIoT-dataset.csv` (read-only).

**Output:** One graph and one JSON audit report. Feature selection comes in Notebook 02.

In [ ]:
from pathlib import Path
from collections import Counter
import hashlib
import json
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = Path(r'D:\J26-CS-336-Network-Flow-V2\network_flow_v2')
DATA = ROOT / 'data/raw/edge_iiotset/ML-EdgeIIoT-dataset.csv'
REPORT = ROOT / 'reports/audits/edge_iiotset_v3_dataset_analysis.json'
FIGURE = ROOT / 'reports/figures/edge_iiotset_v3_class_balance.png'
EXPECTED_ROWS = 157800
CHUNK_SIZE = 30000  # Reads in batches; processes ALL records.
LABELS = ['Attack_label', 'Attack_type']
# Candidates for inspection only; NOT an approved deployment or flow feature set.
CANDIDATES = [
    'tcp.connection.syn', 'tcp.connection.fin', 'tcp.connection.rst',
    'tcp.flags.ack', 'tcp.flags', 'tcp.len', 'tcp.srcport', 'tcp.dstport',
    'udp.time_delta', 'dns.qry.name.len', 'http.content_length',
    'http.request.method', 'mqtt.len', 'mqtt.msgtype',
    'mqtt.topic_len', 'icmp.seq_le'
]
MISSING = {'', '-', '?', 'na', 'n/a', 'nan', 'none', 'null', 'missing', 'unknown'}

if not DATA.is_file():
    raise FileNotFoundError(f'Original CSV not found: {DATA}')
REPORT.parent.mkdir(parents=True, exist_ok=True)
FIGURE.parent.mkdir(parents=True, exist_ok=True)
print('Source:', DATA)
print('All rows will be audited; no sampling or filtering.')

## 1. Columns and labels

Confirm the original schema and candidate columns. They are **packet/protocol fields**, not verified flow statistics.

In [ ]:
columns = pd.read_csv(DATA, nrows=0).columns.tolist()
if len(set(columns)) != len(columns):
    raise ValueError('Duplicate column names: review original CSV header')
missing_required = [c for c in LABELS + CANDIDATES if c not in columns]
if missing_required:
    raise ValueError(f'Required columns absent: {missing_required}')
if len(columns) != 63:
    raise ValueError(f'Expected 63 columns for this selected CSV; found {len(columns)}')
print(f'Original columns: {len(columns)} | Candidate packet/protocol features: {len(CANDIDATES)}')
print('Labels: Attack_label = 0 Normal / 1 Attack; Attack_type = category')
display(pd.DataFrame({'candidate_feature': CANDIDATES}))

## 2. Full dataset check

Count all records and inspect missing/placeholder and zero values. A zero is **not automatically missing**.

In [ ]:
label_counts = Counter()
attack_type_counts = Counter()
missing_counts = Counter()
zero_counts = Counter()
class_missing = { 'Normal': Counter(), 'Attack': Counter() }
row_count = 0
label_type_disagreements = 0

for chunk in pd.read_csv(
    DATA, usecols=CANDIDATES + LABELS, dtype=str,
    keep_default_na=False, chunksize=CHUNK_SIZE, low_memory=False
):
    row_count += len(chunk)
    binary = chunk['Attack_label'].str.strip()
    attack_type = chunk['Attack_type'].str.strip()
    if not binary.isin(['0', '1']).all():
        raise ValueError('Unexpected Attack_label. Expected 0 and 1.')
    if attack_type.eq('').any():
        raise ValueError('Blank Attack_type found; inspect labels before modeling.')
    label_counts.update(binary.value_counts().to_dict())
    attack_type_counts.update(attack_type.value_counts().to_dict())
    is_normal = attack_type.str.casefold().eq('normal')
    label_type_disagreements += int((is_normal != binary.eq('0')).sum())
    for feature in CANDIDATES:
        clean = chunk[feature].astype(str).str.strip().str.casefold()
        missing = clean.isin(MISSING)
        missing_counts[feature] += int(missing.sum())
        zero_counts[feature] += int(clean.isin(['0', '0.0', '0.00']).sum())
        class_missing['Normal'][feature] += int((missing & binary.eq('0')).sum())
        class_missing['Attack'][feature] += int((missing & binary.eq('1')).sum())

if row_count != EXPECTED_ROWS:
    raise ValueError(f'Expected {EXPECTED_ROWS:,} rows but read {row_count:,}. Stop and investigate.')
if label_counts['0'] + label_counts['1'] != row_count:
    raise ValueError('Label totals do not match row count')
if label_type_disagreements:
    raise ValueError(f'{label_type_disagreements} binary/type label disagreements found')

balance = pd.DataFrame([
    {'class': 'Normal', 'records': label_counts['0'], 'percentage': 100 * label_counts['0']/row_count},
    {'class': 'Attack', 'records': label_counts['1'], 'percentage': 100 * label_counts['1']/row_count}
])
quality = pd.DataFrame([
    {'feature': feature,
     'missing_or_placeholder': missing_counts[feature],
     'missing_pct': 100 * missing_counts[feature]/row_count,
     'zero_pct': 100 * zero_counts[feature]/row_count,
     'normal_missing_pct': 100 * class_missing['Normal'][feature]/label_counts['0'],
     'attack_missing_pct': 100 * class_missing['Attack'][feature]/label_counts['1']}
    for feature in CANDIDATES
]).sort_values('missing_pct', ascending=False)
print(f'Processed all records: {row_count:,}/{EXPECTED_ROWS:,} (100%). No rows removed.')
display(balance.round(2))
print('Attack type categories, including Normal:', len(attack_type_counts))
display(pd.DataFrame(attack_type_counts.most_common(), columns=['Attack_type', 'records']))
print('Candidate fields with highest missingness (zero is shown separately):')
display(quality.head(12).round(2))

## 3. Class balance graph

This shows the class distribution, not model accuracy.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
bars = ax.bar(balance['class'], balance['records'])
for bar, rec in zip(bars, balance.to_dict('records')):
    ax.annotate(f"{rec['records']:,} ({rec['percentage']:.2f}%)",
                (bar.get_x() + bar.get_width()/2, bar.get_height()),
                ha='center', xytext=(0, 5), textcoords='offset points')
ax.set_ylim(0, max(balance['records'])*1.2)
ax.set_ylabel('Records')
ax.set_title('Edge-IIoTset — Normal vs Attack')
ax.grid(axis='y', alpha=0.2)
fig.tight_layout()
fig.savefig(FIGURE, dpi=180, bbox_inches='tight')
plt.show()
plt.close(fig)
print('Saved:', FIGURE)

## 4. Save audit

Record counts and source checksum. Notebook 02 will check feature suitability, duplicates and labels in more detail.

In [ ]:
h = hashlib.sha256()
with DATA.open('rb') as f:
    for block in iter(lambda: f.read(8 * 1024 * 1024), b''):
        h.update(block)
report = {
    'dataset': 'Edge-IIoTset',
    'source_file': DATA.name,
    'source_sha256': h.hexdigest(),
    'rows': row_count,
    'columns': len(columns),
    'column_names': columns,
    'candidate_packet_protocol_features': CANDIDATES,
    'target': 'Attack_label',
    'label_columns_excluded_from_features': LABELS,
    'binary_label_counts': dict(label_counts),
    'attack_type_counts': dict(attack_type_counts),
    'label_type_disagreements': label_type_disagreements,
    'feature_missing_or_placeholder_counts': dict(missing_counts),
    'feature_zero_counts': dict(zero_counts),
    'candidate_quality': quality.to_dict(orient='records'),
    'notes': [
        'All source rows read; original CSV unchanged.',
        'These 16 candidates are for screening, not final features.',
        'Selected Edge-IIoTset CSV is packet/protocol-oriented, not verified flow-level data.',
        'No splitting, training or final test performed.'
    ]
}
REPORT.write_text(json.dumps(report, indent=2, ensure_ascii=False), encoding='utf-8')
print('Saved:', REPORT)
print('Next: 02_feature_selection.ipynb')